# T4 (BM25) — tách từ và dựng chỉ mục BM25 trên Kaggle CPU (không tốn hạn mức GPU)

**Cài đặt:** Accelerator = *None* (CPU), Internet = *On* (cài pyvi, jieba).
**Đầu vào:** `mir-code`, `mir-data` (cần `chunks/structure/`).
**Đầu ra:** `/kaggle/working/index/structure/bm25/` → *Save Version* rồi tạo dataset **`mir-bm25`** từ output.
Ước lượng (đo trên máy cá nhân, 1 lõi): pyvi ~340k ký tự/s, jieba ~360k ký tự/s → toàn kho ~1 giờ với 4 tiến trình (chưa đo trên Kaggle).

In [ ]:
# Chuẩn bị chung: chép code, đặt biến môi trường, nối (symlink) dữ liệu đầu vào vào thư mục làm việc
import os, shutil, subprocess, sys, json, glob
CODE_IN = '/kaggle/input/mir-code'          # dataset: mir/, configs/
DATA_IN = '/kaggle/input/mir-data'          # dataset: Data/ (query.parquet, links_corpus.parquet), corpus_clean/, chunks/
WORK = '/kaggle/working'
CODE = f'{WORK}/code'
if not os.path.exists(CODE):
    shutil.copytree(CODE_IN, CODE)
os.chdir(CODE)
os.environ.update(MIR_DATA_DIR=DATA_IN, MIR_WORK_DIR=WORK, HF_HUB_DISABLE_SYMLINKS_WARNING='1')
for name in ('corpus_clean', 'chunks'):
    if not os.path.exists(f'{WORK}/{name}') and os.path.exists(f'{DATA_IN}/{name}'):
        os.symlink(f'{DATA_IN}/{name}', f'{WORK}/{name}')
def sh(cmd):
    print('$', cmd); r = subprocess.run(cmd, shell=True); assert r.returncode == 0, f'lỗi: {cmd}'
sh('pip install -q pyyaml pyvi jieba faiss-cpu')
print(subprocess.run(f'df -h {WORK}; free -g; nproc', shell=True, capture_output=True, text=True).stdout)
CFG = 'configs/baseline.yaml'

In [ ]:
# Bước 1: tách từ từng part (chạy lại được: part đã xong sẽ bỏ qua)
sh(f'python -m mir.index bm25-tokenize --config {CFG} --out {WORK}/index/structure')

In [ ]:
# Bước 2: thống kê toàn cục + ma trận BM25 theo shard; xoá dữ liệu tạm tf/ để dataset nhỏ
sh(f'python -m mir.index bm25-build --config {CFG} --out {WORK}/index/structure')
print(open(f'{WORK}/index/structure/bm25/stats.json').read())
shutil.rmtree(f'{WORK}/index/structure/bm25/tf', ignore_errors=True)
sh(f'du -sh {WORK}/index/structure/bm25')